# 40_01 - ResNet18 Pretrained Transfer Learning

This notebook trains and benchmarks `resnet18_pretrained` for Phase 4.

It is designed to be:

- Run All safe
- portable across Windows and Linux
- safe to rerun after interruption
- safe to rerun without creating duplicate completed artifacts for the same experiment configuration

In [1]:
import json
import os
import random
import sys
from pathlib import Path

import mlflow
import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOTEBOOK_DIR = Path.cwd().expanduser().resolve()


def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "requirements.txt"]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers_all) and any((candidate / marker).exists() for marker in markers_any):
            return candidate
    raise RuntimeError(f"Could not locate project root from: {start}")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
DATA_DIR = PROJECT_ROOT / "data"
CONFIGS_DIR = PROJECT_ROOT / "configs"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
TRACKING_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
SPLIT_DIR = DATA_DIR / "splits" / "split_v1"
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
TEST_CSV = SPLIT_DIR / "test.csv"
CLASSES_JSON = SPLIT_DIR / "classes.json"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset_loader import ImageDataset
from src.data.transforms import load_transforms_config, get_train_transforms, get_eval_transforms
from src.models.cnn_pretrained import (
    TrainingHistory,
    atomic_save_json,
    benchmark_inference,
    build_experiment_signature,
    build_metrics_payload,
    build_model,
    build_training_config,
    collect_device_info,
    configure_trainable_stage,
    count_parameters,
    create_grad_scaler,
    ensure_dir,
    evaluate_model,
    export_model_to_onnx,
    get_model_spec,
    get_trainable_parameter_groups,
    get_weights_name,
    model_size_mb_from_state_dict,
    resolve_run_directory,
    restore_best_weights,
    run_training_stage,
    save_checkpoint_atomic,
    save_report_metrics_copy,
    save_training_curves,
    training_history_from_dict,
)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server


In [3]:
MODEL_NAME = "resnet18_pretrained"
NOTEBOOK_NAME = "40_01_resnet18_pretrained.ipynb"
MODEL_DESCRIPTION = "Lightweight residual pretrained baseline."
BATCH_SIZE = 64
HEAD_ONLY_EPOCHS = 5
FINETUNE_EPOCHS = 15
HEAD_LR = 1e-3
BACKBONE_LR = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.3
GRAD_CLIP_MAX_NORM = 1.0
LR_FACTOR = 0.3
LR_PATIENCE = 2
SEED = 42
SPLIT_ID = "split_v1"
DATASET_VERSION = "prepared_current"
TRANSFORM_ID_TRAIN = "transforms_v1_train_runtime_aug"
TRANSFORM_ID_EVAL = "transforms_v1_eval_resize256_centercrop224_imagenetnorm"
PRETRAINED = True

MODEL_FAMILY_DIR = MODELS_DIR / "cnn_pretrained" / MODEL_NAME
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FAMILY_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

required_paths = {
    "train_csv": TRAIN_CSV,
    "val_csv": VAL_CSV,
    "test_csv": TEST_CSV,
    "classes_json": CLASSES_JSON,
    "transforms_config": TRANSFORMS_CONFIG_PATH,
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs for {MODEL_NAME}: {missing}")

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    classes_to_idx = json.load(f)
NUM_CLASSES = len(classes_to_idx)

cfg = load_transforms_config(TRANSFORMS_CONFIG_PATH)
train_tfm = get_train_transforms(cfg)
eval_tfm = get_eval_transforms(cfg)

train_ds = ImageDataset(
    split_csv=TRAIN_CSV,
    transform=train_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)
val_ds = ImageDataset(
    split_csv=VAL_CSV,
    transform=eval_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)
test_ds = ImageDataset(
    split_csv=TEST_CSV,
    transform=eval_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = DEVICE == "cuda"
NUM_WORKERS = min(8, os.cpu_count() or 2)
PIN_MEMORY = DEVICE == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

device_info = collect_device_info(
    device=DEVICE,
    amp_enabled=AMP_ENABLED,
    num_workers=NUM_WORKERS,
    batch_size=BATCH_SIZE,
)
weights_name = get_weights_name(MODEL_NAME, pretrained=PRETRAINED)
spec = get_model_spec(MODEL_NAME)

training_params = {
    "batch_size": BATCH_SIZE,
    "head_only_epochs": HEAD_ONLY_EPOCHS,
    "finetune_epochs": FINETUNE_EPOCHS,
    "head_lr": HEAD_LR,
    "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY,
    "dropout_p": DROPOUT_P,
    "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
    "lr_factor": LR_FACTOR,
    "lr_patience": LR_PATIENCE,
    "seed": SEED,
    "amp_enabled": AMP_ENABLED,
}

signature_payload = {
    "model_name": MODEL_NAME,
    "weights_name": weights_name,
    "split_id": SPLIT_ID,
    "transform_id_train": TRANSFORM_ID_TRAIN,
    "transform_id_eval": TRANSFORM_ID_EVAL,
    "training_params": training_params,
}
experiment_signature = build_experiment_signature(signature_payload)
resolution = resolve_run_directory(MODEL_FAMILY_DIR, experiment_signature, allow_resume=True)
RUN_ACTION = resolution.action
RUN_DIR = resolution.run_dir

ensure_dir(RUN_DIR)
CHECKPOINT_PATH = RUN_DIR / "checkpoint.pt"
CONFIG_PATH = RUN_DIR / "config.json"
METRICS_PATH = RUN_DIR / "metrics.json"
ONNX_PATH = RUN_DIR / "exported.onnx"

config = build_training_config(
    model_name=MODEL_NAME,
    backbone_name=spec.family,
    weights_name=weights_name,
    split_id=SPLIT_ID,
    transform_id_train=TRANSFORM_ID_TRAIN,
    transform_id_eval=TRANSFORM_ID_EVAL,
    dataset_version=DATASET_VERSION,
    training_params=training_params,
    experiment_signature=experiment_signature,
    extra={
        "device_info": device_info,
        "notebook_name": NOTEBOOK_NAME,
        "model_description": MODEL_DESCRIPTION,
        "run_action": RUN_ACTION,
    },
)

SKIP_TRAINING = RUN_ACTION == "reuse" and CONFIG_PATH.exists() and METRICS_PATH.exists() and CHECKPOINT_PATH.exists()

print("MODEL_NAME          :", MODEL_NAME)
print("RUN_ACTION          :", RUN_ACTION)
print("RUN_DIR             :", RUN_DIR)
print("EXPERIMENT_SIGNATURE:", experiment_signature)
print("DEVICE              :", DEVICE)

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


MODEL_NAME          : resnet18_pretrained
RUN_ACTION          : create
RUN_DIR             : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_pretrained/resnet18_pretrained/run_20260403_103808
EXPERIMENT_SIGNATURE: 1580d749ac8239df
DEVICE              : cuda


In [4]:
if SKIP_TRAINING:
    existing_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    existing_metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    print("[SKIP] Matching completed run already exists:", RUN_DIR)
    print("Existing best epoch:", existing_metrics.get("best_epoch"))
    print("Existing test accuracy:", existing_metrics.get("test_accuracy"))
    FINAL_CONFIG = existing_config
    FINAL_METRICS = existing_metrics
else:
    atomic_save_json(CONFIG_PATH, config)

    model = build_model(
        model_name=MODEL_NAME,
        num_classes=NUM_CLASSES,
        pretrained=PRETRAINED,
        dropout_p=DROPOUT_P,
    ).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    history = TrainingHistory()
    best_state = {}
    skip_head_only = False
    skip_finetune = False

    if RUN_ACTION == "resume" and CHECKPOINT_PATH.exists():
        checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
        if checkpoint.get("experiment_signature") != experiment_signature:
            raise ValueError("Resume checkpoint signature does not match current experiment signature.")
        history = training_history_from_dict(checkpoint.get("history"))
        best_state = {
            key: checkpoint[key]
            for key in [
                "epoch",
                "stage",
                "model_state_dict",
                "optimizer_state_dict",
                "scheduler_state_dict",
                "scaler_state_dict",
                "best_val_macro_f1",
                "best_val_loss",
                "best_val_accuracy",
            ]
            if key in checkpoint
        }
        if "model_state_dict" in best_state:
            model = restore_best_weights(model, best_state)
        completed_stage = checkpoint.get("stage")
        if completed_stage == "head_only":
            skip_head_only = True
        elif completed_stage == "partial_finetune":
            skip_head_only = True
            skip_finetune = True
        print("[RESUME] Found stage checkpoint:", completed_stage)
        print("[RESUME] Epochs already recorded:", len(history.epochs))

    def build_stage_optimizer(stage_name: str):
        configure_trainable_stage(model, MODEL_NAME, stage_name)
        param_groups = get_trainable_parameter_groups(
            model=model,
            model_name=MODEL_NAME,
            head_lr=HEAD_LR,
            backbone_lr=BACKBONE_LR,
            weight_decay=WEIGHT_DECAY,
        )
        optimizer = torch.optim.AdamW(param_groups)
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE)
        scaler = create_grad_scaler(DEVICE, AMP_ENABLED)
        return optimizer, scheduler, scaler

    if HEAD_ONLY_EPOCHS > 0 and not skip_head_only:
        optimizer, scheduler, scaler = build_stage_optimizer("head_only")
        history, best_state = run_training_stage(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=DEVICE,
            num_classes=NUM_CLASSES,
            epochs=HEAD_ONLY_EPOCHS,
            stage_name="head_only",
            history=history,
            best_state=best_state,
            start_epoch=len(history.epochs),
            amp_enabled=AMP_ENABLED,
            scaler=scaler,
            grad_clip_max_norm=GRAD_CLIP_MAX_NORM,
        )
        save_checkpoint_atomic(
            CHECKPOINT_PATH,
            {
                "model_name": MODEL_NAME,
                "experiment_signature": experiment_signature,
                "stage": "head_only",
                "config": config,
                "history": history.to_dict(),
                **best_state,
            },
        )

    if FINETUNE_EPOCHS > 0 and not skip_finetune:
        optimizer, scheduler, scaler = build_stage_optimizer("partial_finetune")
        history, best_state = run_training_stage(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=DEVICE,
            num_classes=NUM_CLASSES,
            epochs=FINETUNE_EPOCHS,
            stage_name="partial_finetune",
            history=history,
            best_state=best_state,
            start_epoch=len(history.epochs),
            amp_enabled=AMP_ENABLED,
            scaler=scaler,
            grad_clip_max_norm=GRAD_CLIP_MAX_NORM,
        )
        save_checkpoint_atomic(
            CHECKPOINT_PATH,
            {
                "model_name": MODEL_NAME,
                "experiment_signature": experiment_signature,
                "stage": "partial_finetune",
                "config": config,
                "history": history.to_dict(),
                **best_state,
            },
        )

    model = restore_best_weights(model, best_state)
    test_metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE,
        num_classes=NUM_CLASSES,
        amp_enabled=AMP_ENABLED,
    )
    benchmark_metrics = benchmark_inference(
        model=model,
        loader=test_loader,
        device=DEVICE,
        warmup_batches=5,
        timed_batches=20,
        amp_enabled=AMP_ENABLED,
    )
    curve_paths = save_training_curves(history, RUN_DIR)

    onnx_export_status = {"attempted": True, "succeeded": False, "path": str(ONNX_PATH), "error": None}
    try:
        export_model_to_onnx(model=model, export_path=ONNX_PATH, device=DEVICE)
        onnx_export_status["succeeded"] = True
    except Exception as exc:
        onnx_export_status["error"] = f"{type(exc).__name__}: {exc}"
        print("[WARNING] ONNX export failed:", onnx_export_status["error"])

    metrics_payload = build_metrics_payload(
        history=history,
        best_state=best_state,
        test_metrics=test_metrics,
        benchmark_metrics=benchmark_metrics,
        parameter_count=count_parameters(model),
        trainable_parameter_count=count_parameters(model, trainable_only=True),
        model_size_mb=model_size_mb_from_state_dict(model),
        device_info=device_info,
        onnx_export_status=onnx_export_status,
    )
    atomic_save_json(METRICS_PATH, metrics_payload)
    report_copy_path = save_report_metrics_copy(REPORTS_METRICS_DIR, MODEL_NAME, RUN_DIR.name, metrics_payload)

    save_checkpoint_atomic(
        CHECKPOINT_PATH,
        {
            "model_name": MODEL_NAME,
            "experiment_signature": experiment_signature,
            "config": config,
            "stage": "partial_finetune" if FINETUNE_EPOCHS > 0 else "head_only",
            "history": history.to_dict(),
            **best_state,
        },
    )

    with mlflow.start_run(run_name=f"{MODEL_NAME}_{RUN_DIR.name}"):
        mlflow.log_param("stage", "cnn_pretrained_training")
        mlflow.log_param("model_name", MODEL_NAME)
        mlflow.log_param("backbone_name", spec.family)
        mlflow.log_param("weights_name", weights_name)
        mlflow.log_param("split_id", SPLIT_ID)
        mlflow.log_param("transform_id_train", TRANSFORM_ID_TRAIN)
        mlflow.log_param("transform_id_eval", TRANSFORM_ID_EVAL)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("head_only_epochs", HEAD_ONLY_EPOCHS)
        mlflow.log_param("finetune_epochs", FINETUNE_EPOCHS)
        mlflow.log_param("head_lr", HEAD_LR)
        mlflow.log_param("backbone_lr", BACKBONE_LR)
        mlflow.log_param("weight_decay", WEIGHT_DECAY)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("device", DEVICE)
        mlflow.log_param("amp_enabled", AMP_ENABLED)
        mlflow.log_param("experiment_signature", experiment_signature)
        mlflow.log_param("run_action", RUN_ACTION)

        mlflow.log_metric("best_val_macro_f1", float(best_state.get("best_val_macro_f1", float("nan"))))
        mlflow.log_metric("best_val_loss", float(best_state.get("best_val_loss", float("nan"))))
        mlflow.log_metric("best_val_accuracy", float(best_state.get("best_val_accuracy", float("nan"))))
        mlflow.log_metric("test_loss", float(test_metrics["loss"]))
        mlflow.log_metric("test_accuracy", float(test_metrics["accuracy"]))
        mlflow.log_metric("test_macro_f1", float(test_metrics["macro_f1"]))
        mlflow.log_metric("latency_ms_per_batch", float(benchmark_metrics["latency_ms_per_batch"]))
        mlflow.log_metric("latency_ms_per_image", float(benchmark_metrics["latency_ms_per_image"]))
        mlflow.log_metric("throughput_img_per_sec", float(benchmark_metrics["throughput_img_per_sec"]))
        mlflow.log_metric("parameter_count", float(count_parameters(model)))
        mlflow.log_metric("model_size_mb", float(model_size_mb_from_state_dict(model)))

        mlflow.log_artifact(str(CONFIG_PATH))
        mlflow.log_artifact(str(METRICS_PATH))
        mlflow.log_artifact(str(CHECKPOINT_PATH))
        mlflow.log_artifact(str(curve_paths["loss_curve"]))
        mlflow.log_artifact(str(curve_paths["accuracy_curve"]))
        if ONNX_PATH.exists():
            mlflow.log_artifact(str(ONNX_PATH))

    print("[OK] Metrics saved:", METRICS_PATH)
    print("[OK] Report metrics copy saved:", report_copy_path)
    print("[OK] Best epoch:", best_state.get("epoch"))
    print("[OK] Test accuracy:", metrics_payload.get("test_accuracy"))

    FINAL_CONFIG = config
    FINAL_METRICS = metrics_payload

print(json.dumps({
    "model_name": MODEL_NAME,
    "run_action": RUN_ACTION,
    "run_dir": str(RUN_DIR),
    "experiment_signature": experiment_signature,
    "best_epoch": FINAL_METRICS.get("best_epoch"),
    "test_accuracy": FINAL_METRICS.get("test_accuracy"),
    "test_macro_f1": FINAL_METRICS.get("test_macro_f1"),
}, indent=2))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/temporaryuser4/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:04<00:00, 9.89MB/s]
/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[head_only] [Epoch 01] train_loss=0.1770 train_acc=0.9350 val_loss=0.0690 val_acc=0.9751 val_f1=0.9749 lr_backbone=nan lr_head=0.001000


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[head_only] [Epoch 02] train_loss=0.1213 train_acc=0.9548 val_loss=0.0588 val_acc=0.9778 val_f1=0.9780 lr_backbone=nan lr_head=0.001000


...

[partial_finetune] [Epoch 09] train_loss=0.0192 train_acc=0.9935 val_loss=0.0188 val_acc=0.9935 val_f1=0.9939 lr_backbone=0.000100 lr_head=0.001000


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[partial_finetune] [Epoch 10] train_loss=0.0160 train_acc=0.9942 val_loss=0.0219 val_acc=0.9938 val_f1=0.9941 lr_backbone=0.000100 lr_head=0.001000
[partial_finetune] [Epoch 11] train_loss=0.0145 train_acc=0.9950 val_loss=0.0226 val_acc=0.9943 val_f1=0.9946 lr_backbone=0.000100 lr_head=0.001000
[partial_finetune] [Epoch 12] train_loss=0.0144 train_acc=0.9952 val_loss=0.0277 val_acc=0.9935 val_f1=0.9938 lr_backbone=0.000030 lr_head=0.000300


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[partial_finetune] [Epoch 13] train_loss=0.0085 train_acc=0.9969 val_loss=0.0208 val_acc=0.9947 val_f1=0.9951 lr_backbone=0.000030 lr_head=0.000300
[partial_finetune] [Epoch 14] train_loss=0.0059 train_acc=0.9979 val_loss=0.0264 val_acc=0.9944 val_f1=0.9947 lr_backbone=0.000030 lr_head=0.000300
[partial_finetune] [Epoch 15] train_loss=0.0054 train_acc=0.9983 val_loss=0.0247 val_acc=0.9938 val_f1=0.9942 lr_backbone=0.000009 lr_head=0.000090
[partial_finetune] [Epoch 16] train_loss=0.0043 train_acc=0.9985 val_loss=0.0236 val_acc=0.9941 val_f1=0.9944 lr_backbone=0.000009 lr_head=0.000090
[partial_finetune] [Epoch 17] train_loss=0.0032 train_acc=0.9988 val_loss=0.0248 val_acc=0.9938 val_f1=0.9942 lr_backbone=0.000009 lr_head=0.000090
[partial_finetune] [Epoch 18] train_loss=0.0035 train_acc=0.9989 val_loss=0.0251 val_acc=0.9943 val_f1=0.9946 lr_backbone=0.000003 lr_head=0.000027
[partial_finetune] [Epoch 19] train_loss=0.0031 train_acc=0.9989 val_loss=0.0265 val_acc=0.9944 val_f1=0.9948 lr